# CA更新の準備

新旧CAを信頼する状態を作る。実行中のジョブを終了してから作業する。


## *Operation Note*

*This is a cell for your own recording. ここに経緯を記述*


# 対象クラスタの設定


In [ ]:
target_group = 'hadoop_all_testcluster'
target_group


In [ ]:
import os
import yaml

with open('group_vars/' + target_group) as stream:
    configuration = yaml.safe_load(stream)


HTTPSで確認するサービスのホスト名とポートを指定する。リスニングポートと現在の設定を確認する。


In [ ]:
!ansible {target_group} --become -a 'ss -lntp'


In [ ]:
https_endpoints = [
    ('test01', 8481), ('test02', 8481),
    ('test01', 50470), ('test02', 50470),
    ('test01', 50475), ('test02', 50475),
    ('test01', 8088), ('test02', 8088),
    ('test01', 8044), ('test02', 8044),
    ('test01', 19890), ('test01', 8190),
]
https_endpoints


# 更新前の確認

利用者として認証し、実行中のYARNアプリケーションがないことを確認する。


In [ ]:
!ansible hadoop_client -l {target_group} -m shell -a 'kinit -kt "$HOME/.hadoop.keytab" "$(id -un)"'


In [ ]:
!ansible hadoop_client -l {target_group} -a 'yarn application -list -appStates NEW,NEW_SAVING,SUBMITTED,ACCEPTED,RUNNING'


NameNodeとJournalNode上でhdfsユーザーとして認証し、HAの状態とDataNode・複製の状態を確認する。


In [ ]:
!ansible hadoop_namenode -l {target_group} --become --become-user hdfs -m shell -a 'kinit -kt /etc/hadoop/conf/hdfs.keytab hdfs/$(hostname -f)'
!ansible hadoop_journalnode -l {target_group} --become --become-user hdfs -m shell -a 'kinit -kt /etc/hadoop/conf/hdfs.keytab hdfs/$(hostname -f)'


In [ ]:
!ansible hadoop_namenode_primary -l {target_group} --become --become-user hdfs -a 'hdfs haadmin -getAllServiceState'
!ansible hadoop_namenode_primary -l {target_group} --become --become-user hdfs -a 'hdfs dfsadmin -report'


# 新しいCAの準備

更新対象CAの公開証明書と、新しいCAを保管するディレクトリを指定する。新しいCAの鍵は新規に作成する。


In [ ]:
old_ca_certificate_file = os.path.expanduser('~/.local/share/hadoop-pki/initial/ca.crt')
pki_dir = os.path.expanduser('~/.local/share/hadoop-pki/replacement')
(old_ca_certificate_file, pki_dir)


In [ ]:
!openssl x509 -in "{old_ca_certificate_file}" -noout -text -fingerprint -sha256
!openssl x509 -in "{old_ca_certificate_file}" -noout -checkend 0


In [ ]:
ca_subject = '/CN=Hadoop Root CA replacement'
ca_days = 3650
certificate_days = 365
(ca_subject, ca_days, certificate_days)


In [ ]:
ca_key_file = os.path.join(pki_dir, 'ca.key')
ca_certificate_file = os.path.join(pki_dir, 'ca.crt')
truststore_file = os.path.join(pki_dir, 'truststore.jks')
(ca_key_file, ca_certificate_file, truststore_file)


In [ ]:
!openssl version
!keytool -help


CA秘密鍵のパスワードを入力する。ホストのkeystoreとtruststoreには、group_varsに指定したパスワードを使用する。


In [ ]:
import getpass

os.environ['HADOOP_CA_PASSWORD'] = getpass.getpass('CA key password: ')
os.environ['HADOOP_KEYSTORE_PASSWORD'] = configuration['https_keystore_pass']
os.environ['HADOOP_PRIVATE_KEY_PASSWORD'] = configuration['https_privkey_pass']
os.environ['HADOOP_TRUSTSTORE_PASSWORD'] = configuration['https_truststore_pass']


In [ ]:
!install -d -m 0700 "{pki_dir}"
!test ! -e "{ca_key_file}"
!test ! -e "{ca_certificate_file}"


ルートCAの鍵と自己署名証明書を作成する。


In [ ]:
!umask 077; openssl req -new -x509 -newkey rsa:3072 -sha256 -days {ca_days} \
    -subj "{ca_subject}" \
    -addext "basicConstraints=critical,CA:TRUE,pathlen:0" \
    -addext "keyUsage=critical,keyCertSign,cRLSign" \
    -passout env:HADOOP_CA_PASSWORD \
    -keyout "{ca_key_file}" \
    -out "{ca_certificate_file}"


CAのSubject、有効期限、CAとしての制約とフィンガープリントを確認する。


In [ ]:
!openssl x509 -in "{ca_certificate_file}" -noout -text -fingerprint -sha256


# Playbookの準備


In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
work_dir


In [ ]:
!git clone https://github.com/NII-cloud-operation/Literate-computing-Hadoop.git {work_dir}/hadoop
!git -C {work_dir}/hadoop log -1


In [ ]:
playbook_dir = os.path.join(work_dir, 'hadoop/playbooks')
playbook_dir


# 新旧CAを信頼する設定

新しい証明書を配る前に、すべてのノードへ新旧CAを含むtruststoreを配布する。

クラスタ外のクライアントにも、新旧CAを信頼する設定を反映して接続を確認する。


In [ ]:
!keytool -importcert -noprompt -alias root-ca -file "{ca_certificate_file}" \
    -keystore "{truststore_file}" -storetype JKS \
    -storepass:env HADOOP_TRUSTSTORE_PASSWORD


In [ ]:
!keytool -importcert -noprompt -alias old-root-ca -file "{old_ca_certificate_file}" \
    -keystore "{truststore_file}" \
    -storepass:env HADOOP_TRUSTSTORE_PASSWORD


In [ ]:
!keytool -list -v -keystore "{truststore_file}" -storepass:env HADOOP_TRUSTSTORE_PASSWORD


In [ ]:
ca_bundle_file = os.path.join(pki_dir, 'ca-certificates.pem')
ca_bundle_file


In [ ]:
!cat "{old_ca_certificate_file}" "{ca_certificate_file}" > "{ca_bundle_file}"


In [ ]:
https_keystore_path = configuration['https_keystore_path']
https_truststore_path = configuration['https_truststore_path']
ca_bundle_path = '/etc/hadoop/conf/ca-certificates.pem'
(https_keystore_path, https_truststore_path, ca_bundle_path)


In [ ]:
next_ca_path = '/etc/hadoop/conf/ca-next.crt'
next_ca_path


In [ ]:
!ansible {target_group} --become -m copy -a 'src={truststore_file} dest={https_truststore_path} owner=root group=root mode=0444'
!ansible {target_group} --become -m copy -a 'src={ca_bundle_file} dest={ca_bundle_path} owner=root group=root mode=0444'
!ansible {target_group} --become -m copy -a 'src={ca_certificate_file} dest={next_ca_path} owner=root group=root mode=0444'


HTTPSサービスを1台ずつ再起動してtruststoreを読み直す。NameNodeはJournalNodeからHTTPSでログを取得するため、JournalNodeを先に再起動する。JournalNodeは1台ごとにログをロールして書き込みへ復帰させてから、次のサーバーへ進む。


In [ ]:
!ansible-playbook {playbook_dir}/restart_https_services.yml -l {target_group}


新旧CAを含む設定で、現在の証明書とのTLS接続が検証できることを確認する。


In [ ]:
for hostname, port in https_endpoints:
    !ansible hadoop_client -l {target_group} -m shell -a 'openssl s_client -brief -verify_return_error -verify_hostname {hostname} -CAfile {ca_bundle_path} -connect {hostname}:{port} < /dev/null'


各ホストでO12mを実行する。全ホストの切り替えが済んだらO12nへ進む。


In [ ]:
del os.environ['HADOOP_CA_PASSWORD']
del os.environ['HADOOP_KEYSTORE_PASSWORD']
del os.environ['HADOOP_PRIVATE_KEY_PASSWORD']
del os.environ['HADOOP_TRUSTSTORE_PASSWORD']
